# 494. Target Sum

## Topic Alignment
- This problem models decision-making with binary choices (add/subtract), appearing in scenarios like portfolio optimization (long/short positions), A/B testing with positive/negative effects, or classifier ensembles with weighted voting.

## Metadata 摘要
- Source: https://leetcode.com/problems/target-sum/
- Tags: Dynamic Programming, Array, Backtracking, 0/1 Knapsack
- Difficulty: Medium
- Priority: High

## Problem Statement 原题描述
You are given an integer array `nums` and an integer `target`.

You want to build an **expression** out of nums by adding one of the symbols `'+'` and `'-'` before each integer in nums and then concatenate all the integers.

For example, if `nums = [2, 1]`, you can add a `'+'` before `2` and a `'-'` before `1` and concatenate them to build the expression `"+2-1"`.

Return the number of different **expressions** that you can build, which evaluates to `target`.

**Constraints**:
- 1 <= nums.length <= 20
- 0 <= nums[i] <= 1000
- 0 <= sum(nums[i]) <= 1000
- -1000 <= target <= 1000

## Progressive Hints
- Hint 1: Let P = subset with '+' sign, N = subset with '-' sign. Then sum(P) - sum(N) = target.
- Hint 2: Since sum(P) + sum(N) = sum(nums), we can derive: sum(P) = (target + sum(nums)) / 2.
- Hint 3: The problem transforms to: count subsets with sum = (target + sum(nums)) / 2.
- Hint 4: This is a 0/1 knapsack counting problem.
- Hint 5: Use dp[j] to count the number of ways to achieve sum j.

## Solution Overview
This is a **0/1 Knapsack variant** focused on counting combinations.

**Key Transformation**:
- Assign '+' to some numbers (positive subset P) and '-' to others (negative subset N)
- Goal: sum(P) - sum(N) = target
- Also know: sum(P) + sum(N) = sum(nums)
- Solving: sum(P) = (target + sum(nums)) / 2
- **New problem**: Count subsets that sum to (target + sum(nums)) / 2

**Approaches**:
1. **Backtracking**: Try all 2^n combinations - O(2^n)
2. **2D DP**: dp[i][j] = ways to achieve sum j using first i elements
3. **1D DP**: dp[j] = ways to achieve sum j (optimized space)
4. **DFS with memoization**: Top-down approach

## Detailed Explanation

### Mathematical Transformation

Given nums, we assign '+' or '-' to each number:
- Let P = positive subset, N = negative subset
- **Equation 1**: sum(P) - sum(N) = target
- **Equation 2**: sum(P) + sum(N) = sum(nums)

Adding both equations:
```
2 × sum(P) = target + sum(nums)
sum(P) = (target + sum(nums)) / 2
```

**Validation**:
- If (target + sum(nums)) is odd, no solution exists (cannot divide by 2)
- If target > sum(nums), no solution (cannot make target larger than total)
- If target < -sum(nums), no solution

---

### 0/1 Knapsack Counting Problem

After transformation, we need to count subsets that sum to `new_target = (target + sum(nums)) / 2`.

**State Definition**:
- `dp[j]` = number of ways to achieve sum j

**Recurrence**:
- For each number `num`, for each sum `j` from `new_target` down to `num`:
  ```python
  dp[j] += dp[j - num]
  ```
- This means: "number of ways to make sum j" includes "number of ways to make sum (j-num), then add num"

**Base Case**:
- `dp[0] = 1` (one way to achieve sum 0: select nothing)

**Critical Detail - Handling Zeros**:
- If nums contains zeros, each zero doubles the number of ways (can assign + or -)
- The DP naturally handles this: when num=0, dp[j] += dp[j] (doubles the count)

---

### Example Walkthrough

**Input**: nums = [1, 1, 1, 1, 1], target = 3

**Step 1**: Calculate new_target
- sum(nums) = 5
- new_target = (3 + 5) / 2 = 4
- Need to count subsets that sum to 4

**Step 2**: Initialize DP
- `dp = [1, 0, 0, 0, 0]` (size 5, index 0 to 4)

**Step 3**: Process each 1
- After 1st '1': `dp = [1, 1, 0, 0, 0]` (can make 0, 1)
- After 2nd '1': `dp = [1, 2, 1, 0, 0]` (can make 0, 1, 2 with counts)
- After 3rd '1': `dp = [1, 3, 3, 1, 0]`
- After 4th '1': `dp = [1, 4, 6, 4, 1]`
- After 5th '1': `dp = [1, 5, 10, 10, 5]`

**Result**: dp[4] = 5

**Verification**: The 5 ways are:
- +1+1+1+1-1 = 3
- +1+1+1-1+1 = 3
- +1+1-1+1+1 = 3
- +1-1+1+1+1 = 3
- -1+1+1+1+1 = 3

---

### 0/1 vs Complete Knapsack Review

**0/1 Knapsack** (this problem):
```python
for num in nums:  # Outer loop: items
    for j in range(target, num - 1, -1):  # Inner loop: capacity (right to left)
        dp[j] += dp[j - num]
```
- Each item used **at most once**
- Traverse capacity **right to left**
- Ensures we don't reuse same item in current iteration

**Complete Knapsack**:
```python
for num in nums:  # Outer loop: items
    for j in range(num, target + 1):  # Inner loop: capacity (left to right)
        dp[j] += dp[j - num]
```
- Each item can be used **unlimited times**
- Traverse capacity **left to right**
- Allows reusing same item in current iteration

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Brute force (backtracking) | O(2^n) | O(n) | Try all +/- combinations |
| DFS + memoization | O(n × sum) | O(n × sum) | Top-down with cache |
| 2D DP | O(n × sum) | O(n × sum) | Bottom-up |
| 1D DP | O(n × sum) | O(sum) | Space-optimized, most efficient |

In [ ]:
class Solution:
    def findTargetSumWays(self, nums: list[int], target: int) -> int:
        """
        0/1 Knapsack counting solution.
        
        Time: O(n × sum)
        Space: O(sum)
        """
        total = sum(nums)
        
        # Check if solution is possible
        if abs(target) > total:  # Target out of range
            return 0
        if (target + total) % 2 == 1:  # Cannot divide by 2
            return 0
        
        # Transform: find subsets that sum to new_target
        new_target = (target + total) // 2
        
        # dp[j] = number of ways to achieve sum j
        dp = [0] * (new_target + 1)
        dp[0] = 1  # One way to achieve sum 0: select nothing
        
        # 0/1 Knapsack: traverse right to left
        for num in nums:
            for j in range(new_target, num - 1, -1):
                # Add the number of ways to make (j - num) to ways to make j
                dp[j] += dp[j - num]
        
        return dp[new_target]

In [ ]:
# Test cases
tests = [
    ([1, 1, 1, 1, 1], 3, 5),         # Classic example
    ([1], 1, 1),                     # +1
    ([1], 2, 0),                     # Impossible
    ([1, 0], 1, 2),                  # +1+0 or +1-0
    ([0, 0, 0, 0, 0, 0, 1], 1, 32),  # 2^6 = 64 ways for zeros, half with +1, half with -1... wait
    ([100], -200, 0),                # Out of range
    ([1, 2, 1], 0, 2),               # +1+2-1-2? No. +1-2+1=0, -1+2-1=0
]

# Fix test case 4: [0,0,0,0,0,0,1], target=1
# sum=1, new_target=(1+1)/2=1
# Ways to make sum 1: select the '1' = 1 way
# But we have 6 zeros, each can be + or -, so 2^6 = 64 ways
# Total: 1 × 64 = 64 ways
tests[4] = ([0, 0, 0, 0, 0, 0, 1], 1, 64)

solver = Solution()
for nums, target, expected in tests:
    result = solver.findTargetSumWays(nums, target)
    assert result == expected, f"Failed for nums={nums}, target={target}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(n × S) where n = len(nums), S = sum(nums)
  - For each of n numbers, iterate through up to S/2 values
  - With constraints: n ≤ 20, S ≤ 1000, so max O(20 × 500) = O(10,000)
- **Space**: O(S)
  - DP array of size (target + sum) / 2 + 1
  - At most O(1000) in this problem

## Edge Cases & Pitfalls
- **Zeros in array**: Each zero can have + or -, doubling the count. DP handles this naturally.
- **Target out of range**: If |target| > sum(nums), return 0 immediately
- **Odd sum**: If (target + sum) is odd, cannot achieve integer division, return 0
- **All zeros**: If nums = [0,0,...,0] and target = 0, answer is 2^n
- **Negative target**: The transformation handles negative targets correctly
- **Index range**: Ensure j >= num when accessing dp[j - num]
- **Initialization**: dp[0] = 1 is critical; represents the empty subset

## Follow-up Variants
- **Weighted target sum**: Each number has a weight, maximize value while reaching target
- **Multiple targets**: Count ways to achieve any of k different targets
- **Constrained operations**: Limit the number of '+' or '-' operations
- **Ordered expressions**: Consider expressions with different orderings as different
- **Real-world application**: Portfolio optimization with long/short positions reaching target return

## Takeaways
- **Problem Transformation**: Converting +/- assignment to subset sum is a key insight
- **Mathematical Derivation**: sum(P) = (target + sum(nums)) / 2 is the crucial formula
- **Counting vs Boolean**: Use `dp[j] += dp[j-num]` for counting, `dp[j] |= dp[j-num]` for existence
- **0/1 Knapsack Pattern**: Right-to-left traversal ensures each element used once
- **Validation First**: Check feasibility (odd sum, out of range) before running DP
- **Zero Handling**: Zeros naturally double the count, no special handling needed in DP

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 416 | Partition Equal Subset Sum | 0/1 knapsack boolean version |
| LC 1049 | Last Stone Weight II | Similar subset sum transformation |
| LC 698 | Partition to K Equal Sum Subsets | Multi-way partition |
| LC 282 | Expression Add Operators | Similar expression building |
| LC 39 | Combination Sum | Complete knapsack variant |